# Grad-CAM and ROI Analysis

Before running the notebook, update the configuration values to match your environment.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install face_alignment

In [ ]:
from pathlib import Path
import glob
import os
import sys

import cv2
import face_alignment
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from tqdm import tqdm

#############################################################################
# Configuration
# Update the configuration below before running this notebook.
# Example: PROJECT_ROOT = Path('/content/drive/MyDrive/DE-Fake-it')
#############################################################################

PROJECT_ROOT = Path('/content/drive/MyDrive/DE-Fake-it')
DATA_ROOT = PROJECT_ROOT / 'Dataset'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
TEST_DATA_ROOT = DATA_ROOT / 'ff++' / 'test'
BEFORE_DATA_ROOT = TEST_DATA_ROOT
GRADCAM_DATA_ROOT = DATA_ROOT / 'ff++(grad-cam)'

MODEL_NAME = 'EfficientNet-b0'
CHECKPOINT_NAME = 'checkpoint_v35'
BEST_CHECKPOINT_NAME = CHECKPOINT_NAME + '_best'
CHECKPOINT_PATH = str(CHECKPOINT_DIR / CHECKPOINT_NAME)
PREDICTIONS_FILE_PATH = str(CHECKPOINT_DIR / CHECKPOINT_NAME / f'(test)_{CHECKPOINT_NAME}_predictions_ff++.xlsx')

# For quick tests, include only folders that actually exist under BEFORE_DATA_ROOT; empty folders are skipped by Grad-CAM but can leave ROI results empty.
FOLDER_LIST = ["Deepfakes", "original", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]
FOLDER_INPUT_PATHS = {
    "Deepfakes": BEFORE_DATA_ROOT / "fake" / "Deepfakes",
    "original": BEFORE_DATA_ROOT / "real" / "original",
    "Face2Face": BEFORE_DATA_ROOT / "fake" / "Face2Face",
    "FaceShifter": BEFORE_DATA_ROOT / "fake" / "FaceShifter",
    "FaceSwap": BEFORE_DATA_ROOT / "fake" / "FaceSwap",
    "NeuralTextures": BEFORE_DATA_ROOT / "fake" / "NeuralTextures",
}

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Configuration")
print(f"Model: {MODEL_NAME}")
print("Dataset: FaceForensics++ Grad-CAM")
print(f"Before data root: {BEFORE_DATA_ROOT}")
print(f"Grad-CAM data root: {GRADCAM_DATA_ROOT}")
print(f"Checkpoint: {BEST_CHECKPOINT_NAME}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Predictions file: {PREDICTIONS_FILE_PATH}")
print(f"Folders: {', '.join(FOLDER_LIST)}")
for folder_name, input_path in FOLDER_INPUT_PATHS.items():
    print(f"Input[{folder_name}]: {input_path}")


## Load Model


In [ ]:
from model.research.device import get_device
from model.research.modeling import ResearchModel as Model

device = get_device()
print(f"Using device: {device}")

model = Model(num_binary_classes=2, num_method_classes=7, model_name=MODEL_NAME).to(device)
model.load_state_dict(torch.load(f'{CHECKPOINT_PATH}/{BEST_CHECKPOINT_NAME}.pt', map_location=device))
model.eval()
model.lstm.train()


## Grad-CAM Utilities


In [ ]:
# ✅ Grad-CAM computation for binary classification
def compute_gradcam_binary(model, input_tensor, device=None, target_class=0):
    fmap = None
    grad = None
    device = device or input_tensor.device

    def fw_hook(module, inp, out):
        nonlocal fmap
        fmap = out.detach()

    def bw_hook(module, grad_in, grad_out):
        nonlocal grad
        grad = grad_out[0].detach()

    last_layer = model.model[-1]
    f = last_layer.register_forward_hook(fw_hook)
    b = last_layer.register_backward_hook(bw_hook)

    input_tensor = input_tensor.to(device).unsqueeze(0).unsqueeze(0).requires_grad_(True)
    _, binary_output, method_output = model(input_tensor)

    # Get the probability of the target class
    prob = F.softmax(binary_output, dim=1)[0, target_class].item()

    # Predict binary and method classes
    binary_pred = torch.argmax(binary_output, dim=1).item()   # 0: fake, 1: real
    method_pred = torch.argmax(method_output, dim=1).item()   # 0: original, 1~6: fake methods, 7: others

    # 🔴 Condition 1: If the prediction is real and the method is original, skip CAM computation / 조건 1: real(1) + original(0) → CAM X
    if binary_pred == 1 and method_pred == 0:
        cam = np.zeros((input_tensor.shape[-2], input_tensor.shape[-1]))
        f.remove()
        b.remove()
        return cam, prob

    # Grad-CAM for fake class (target_class = 0)
    target_class = 0
    model.zero_grad()
    binary_output[0, target_class].backward()

    # Compute Grad-CAM
    weights = grad.mean(dim=[2, 3], keepdim=True)
    cam = (weights * fmap).sum(dim=1, keepdim=True)
    cam = F.relu(cam).squeeze().cpu().numpy()

    # 🔵 Condition 2: If the prediction is fake and the method is not original, enhance CAM / 조건 2: fake (0) + method (1~7) (≠ 0) → CAM ↑
    if binary_pred == 0 and method_pred != 0:
        cam *= 1.5

    # Normalize and resize the CAM
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam = cv2.resize(cam, (input_tensor.shape[-1], input_tensor.shape[-2]))

    f.remove()
    b.remove()
    return cam, prob

# ✅ Function to process video, compute Grad-CAM, and save frames
def process_video_and_save_frames(input_video_path, output_video_path, model, frame_dir, device, predictions_file_path):
    input_path = f'{input_video_path}/*.mp4'
    video_files = glob.glob(input_path)
    already_present_count = glob.glob(output_video_path+ '/*.mp4')
    print(len(video_files))
    print("No of videos already present ", len(already_present_count))

    # Create output folders if not exist
    os.makedirs(frame_dir, exist_ok=True)
    os.makedirs(output_video_path, exist_ok=True)
    top_jpg_dir = os.path.join(output_video_path, "Top_jpg")
    os.makedirs(top_jpg_dir, exist_ok=True)

    # Load REAL/FAKE labels
    df = pd.read_excel(predictions_file_path)
    for video_file in tqdm(video_files):
        file_name=os.path.basename(video_file)
        result = str(df[df['Filename'] == file_name]['label'].iloc[0])[0] + str(df[df['Filename'] == file_name]['Prediction'].iloc[0])[0]+str(df[df['Filename'] == file_name]['Predicted_method'].iloc[0])

        f_name=f'({result})_'+file_name
        out_path = os.path.join(output_video_path,f_name) # Extract output video file name
        file_exists = glob.glob(out_path + "*")

        if(len(file_exists) != 0): # Skip if video already exists
            print("File Already exists: " , out_path)
            continue

        cap = cv2.VideoCapture(video_file)
        fps = cap.get(cv2.CAP_PROP_FPS)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # Set output MP4 file name
        name=os.path.splitext(file_name)[0]
        out = cv2.VideoWriter(out_path,cv2.VideoWriter_fourcc('M','J','P','G'), fps, (w, h))

        # Preprocessing transforms
        transform = T.Compose([
            T.ToPILImage(),
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406],
                                [0.229, 0.224, 0.225])
        ])

        frame_count = 0

        frame_scores = []
        frame_images = []
        roi_result=[]

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            original = frame.copy()
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = transform(img).to(device)

            # 1️⃣ Compute Grad-CAM for binary classification
            cam, score = compute_gradcam_binary(model, img, device)
            
            # Generate Grad-CAM overlay for the frame
            heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
            heatmap = cv2.resize(heatmap, (original.shape[1], original.shape[0]))
            overlay = 0.4 * heatmap + 0.6 * original
            overlay = np.clip(overlay, 0, 255).astype(np.uint8)

            frame_scores.append((frame_count, score))
            frame_images.append((frame_count, overlay))

            out.write(overlay) # Save Grad-CAM overlay video

            # Save the Grad-CAM overlay frame as an image
            frame_path = os.path.join(frame_dir, f"({result})_{name}_{frame_count:04d}_{score:10f}.jpg")
            cv2.imwrite(frame_path, overlay)
            frame_count += 1

        # ✅ Save top 10 frames with highest scores(probability)
        frame_scores.sort(key=lambda x: x[1], reverse=True)
        top_10_indices = [idx for idx, score in frame_scores[:10]]

        for rank, idx in enumerate(top_10_indices):
            img = frame_images[idx][1]
            score = frame_scores[rank][1]

            top_frame_path = os.path.join(top_jpg_dir, f"{name}_TOP{rank + 1}_Score{score:10f}.jpg")
            cv2.imwrite(top_frame_path, img)   # Save top 10 frames

        cap.release()  # Save frames
        out.release()  # Save videos

        print(f"✅ Grad-CAM video saved: {output_video_path}")
        print(f"✅ {frame_count} frame images saved: {frame_dir}/frame_XXXX.jpg")


## Generate Grad-CAM Videos and Frames


In [ ]:
for folder_name in FOLDER_LIST:
    input_video_path = str(FOLDER_INPUT_PATHS[folder_name])
    output_video_path = str(GRADCAM_DATA_ROOT / folder_name / 'video')
    frame_path = str(GRADCAM_DATA_ROOT / folder_name / 'frame')

    process_video_and_save_frames(
        input_video_path,
        output_video_path,
        model=model,
        frame_dir=frame_path,
        device=device,
        predictions_file_path=PREDICTIONS_FILE_PATH,
    )


## ROI Utilities


In [ ]:
def get_bbox(pts):
    x, y = pts[:,0], pts[:,1]
    return int(x.min()), int(y.min()), int(x.max()), int(y.max())

def roi_activation(cam, bbox):
    x1, y1, x2, y2 = bbox
    patch = cam[y1:y2, x1:x2]
    mean_val = float(patch.mean())

    if np.isnan(mean_val):
        return -1
    return mean_val

def analyze_roi_activation(input_dir, output_dir_box, base_path, folder_name, model, device):

  fa = face_alignment.FaceAlignment(
      face_alignment.LandmarksType.TWO_D,
      device=str(device)
  )

  os.makedirs(output_dir_box, exist_ok=True)

  transform = T.Compose([
      T.ToTensor(),
      T.Resize((224, 224)),
      T.Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225])
  ])

  result = []
  video_result = []
  prediction_df = pd.read_excel(PREDICTIONS_FILE_PATH) if os.path.exists(PREDICTIONS_FILE_PATH) else pd.DataFrame()
  method_class_names = {
      0: "original",
      1: "Deepfakes",
      2: "FaceShifter",
      3: "FaceSwap",
      4: "NeuralTextures",
      5: "Face2Face",
      6: "others",
  }

  video_paths = glob.glob(os.path.join(input_dir, '*.mp4'))
  if not video_paths:
    print(f"No videos found for {folder_name}: {input_dir}. Skipping ROI analysis.")
    return

  for video_path in tqdm(video_paths):
    facial_region=['jawline', 'left_eye', 'right_eye', 'left_eye_brow', 'right_eye_brow', 'nose', 'mouth','None']
    first_detection_count = {key: 0 for key in facial_region}
    second_detection_count = {key: 0 for key in facial_region}
    detection_probability={key: 0.0 for key in facial_region}

    cap = cv2.VideoCapture(video_path)
    frame_idx = 0
    video_file_name = os.path.basename(video_path)
    video_name = os.path.splitext(video_file_name)[0]
    binary_pred = None
    final_probability = None
    method_pred = folder_name
    if not prediction_df.empty and 'Filename' in prediction_df.columns:
      prediction_row = prediction_df[prediction_df['Filename'] == video_file_name]
      if not prediction_row.empty:
        if 'Prediction' in prediction_df.columns:
          binary_pred = prediction_row['Prediction'].iloc[0]
        if 'Probability' in prediction_df.columns:
          final_probability = prediction_row['Probability'].iloc[0]
        if 'Predicted_method' in prediction_df.columns:
          method_id = prediction_row['Predicted_method'].iloc[0]
          if pd.notna(method_id):
            method_pred = method_class_names.get(int(method_id), "others")
    if binary_pred is None:
      binary_pred = "REAL" if folder_name == "original" else "FAKE"

    while cap.isOpened():
      success, frame = cap.read()
      if not success:
        break

      rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
      landmarks = fa.get_landmarks(rgb)
      if not landmarks:
        frame_idx += 1
        continue
      lm = landmarks[0]

      # ROI BBoxes
      bbox_map = {
          'jawline': get_bbox(lm[0:17]),
          'left_eye': get_bbox(lm[36:42]),
          'right_eye': get_bbox(lm[42:48]),
          'left_eye_brow': get_bbox(lm[17:22]),
          'right_eye_brow': get_bbox(lm[22:27]),
          'nose': get_bbox(lm[27:36]),
          'mouth': get_bbox(lm[48:68]),
      }

      # Grad-CAM
      img = transform(rgb).to(device)
      cam, frame_cam_score = compute_gradcam_binary(model, img, device)
      cam = cv2.resize(cam, (frame.shape[1], frame.shape[0]))

      scores = {region: roi_activation(cam, box) for region, box in bbox_map.items()}
      sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
      first_activated_region = sorted_scores[0][0]
      second_activated_region = sorted_scores[1][0]

      f_x1, f_y1, f_x2, f_y2 = bbox_map[first_activated_region]
      s_x1, s_y1, s_x2, s_y2 = bbox_map[second_activated_region]

      # 1️⃣ Grad-CAM 히트맵 오버레이
      heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
      overlay = cv2.addWeighted(frame, 0.6, heatmap, 0.4, 0)

      # 2️⃣ 원본 프레임 복사해서 박스용 준비
      frame_with_box = frame.copy()
      overlay_with_box = overlay.copy()

      # 3️⃣ 박스 그리기
      if scores[first_activated_region] > 0:
        cv2.rectangle(frame_with_box, (f_x1, f_y1), (f_x2, f_y2), (0, 255, 0), 2)
        cv2.rectangle(overlay_with_box, (f_x1, f_y1), (f_x2, f_y2), (0, 255, 0), 2)
      else:
        first_activated_region="None"

      if scores[second_activated_region] > 0:
        cv2.rectangle(frame_with_box, (s_x1, s_y1), (s_x2, s_y2), (255, 0, 0), 2)
        cv2.rectangle(overlay_with_box, (s_x1, s_y1), (s_x2, s_y2), (255, 0, 0), 2)
      else:
        second_activated_region="None"

      # 4️⃣ 파일 저장
      file_id = f"{video_name}_frame{frame_idx:04d}"
      cv2.imwrite(os.path.join(output_dir_box, f"{file_id}_roi.jpg"), frame_with_box)
      cv2.imwrite(os.path.join(output_dir_box, f"{file_id}_gradcam.jpg"), overlay_with_box)

      first_detection_count[first_activated_region]+=1
      second_detection_count[second_activated_region]+=1
      for key in facial_region:
        if key!="None" and scores[key]!=-1:
          detection_probability[key]+=frame_cam_score * scores[key]

      result.append({
          'file_name': file_id,
          'cam_score': frame_cam_score,
          'first_activate_region': first_activated_region,
          'second_activate_region': second_activated_region,
          'f_x1': f_x1, 'f_y1': f_y1, 'f_x2': f_x2, 'f_y2': f_y2,
          's_x1': s_x1, 's_y1': s_y1, 's_x2': s_x2, 's_y2': s_y2,
          **scores,
      })

      frame_idx += 1

    # Printing all the dictionaries
    first_detection_rate = {key: round((value / frame_idx)*100, 2) for key, value in first_detection_count.items()}
    second_detection_rate = {key: round((value / frame_idx)*100, 2) for key, value in second_detection_count.items()}

    # 📌 Note: A high proportion of 'None' may inflate the relative Contribution (%) and should be interpreted with caution.
    raw_detection_probability= {key: round(value, 4) for key, value in detection_probability.items()}
    probability_total = sum(detection_probability.values())
    detection_probability= {key: round((value/probability_total)*100, 2) for key, value in detection_probability.items()}

    print("Video name:", video_name)
    print("Facial Region:", facial_region)
    print("First Detection Count:", first_detection_count)
    print("Second Detection Count:", second_detection_count)
    print("First Detection Rate:", first_detection_rate)
    print("Second Detection Rate:", second_detection_rate)
    print("Raw_Detection Probability:", raw_detection_probability)
    print("Detection Probability:", detection_probability)

    video_result.append({
        'video_name': video_name,
        'binary_pred': binary_pred,
        'final_probability': final_probability,
        'method_pred': method_pred,
        'facial_region': facial_region,
        'first_detection_count': first_detection_count,
        'second_detection_count': second_detection_count,
        'first_detection_rate': first_detection_rate,
        'second_detection_rate': second_detection_rate,
        'raw_detection_probability': raw_detection_probability,
        'detection_probability': detection_probability,
    })

  # 저장
  df = pd.DataFrame(result)
  df = df[[
      'file_name', 'cam_score',
      'first_activate_region', 'second_activate_region',
      'f_x1', 'f_y1', 'f_x2', 'f_y2',
      's_x1', 's_y1', 's_x2', 's_y2',
      'jawline', 'left_eye', 'right_eye',
      'left_eye_brow', 'right_eye_brow',
      'nose', 'mouth',
  ]]

  df.to_excel(f"{base_path}/{folder_name}_frame_roi_.xlsx", index=False)

  video_df = pd.DataFrame(video_result)
  video_df.to_excel(f"{base_path}/{folder_name}_video_roi_result.xlsx", index=False)
  video_df.to_json(f"{base_path}/{folder_name}_video_roi_result.json", orient='records', force_ascii=False)


## Analyze ROI Activation


In [ ]:
for folder_name in FOLDER_LIST:
    input_dir = str(FOLDER_INPUT_PATHS[folder_name])
    output_dir_box = str(GRADCAM_DATA_ROOT / folder_name / 'roi_frame')
    base_path = str(GRADCAM_DATA_ROOT / folder_name)

    analyze_roi_activation(
        input_dir,
        output_dir_box,
        base_path,
        folder_name,
        model=model,
        device=device,
    )
